# 🐍 Day 4: Redis

- Redis basics
- Caching patterns
- Sessions
- Rate limiting

## Why Redis?

Redis is an in-memory key-value store. Fast for caching, sessions, rate limiting, pub/sub, and queues.

In [ ]:
# pip install redis
import redis

r = redis.Redis(host="localhost", port=6379, db=0, decode_responses=True)
# r.set("key", "value")
# r.get("key")  # "value"
# r.delete("key")
# r.exists("key")  # 0 or 1
print("redis.Redis(host, port, decode_responses=True)")

## Basic Operations

Strings: set, get, setex (with TTL). Use setex for cache expiry.

In [ ]:
# Simulate without Redis (for environments without Redis)
cache = {}

def redis_set(key, value, ex=None):
    cache[key] = value
    # ex = seconds to expire

def redis_get(key):
    return cache.get(key)

redis_set("user:1", "Alice")
print("get:", redis_get("user:1"))
print("Real Redis: r.set('k','v', ex=3600) for 1hr TTL")

## Caching Pattern

Check cache first. If miss, compute, store, return.

In [ ]:
import json

def get_cached(key, fetch_fn, ttl=300):
    # cached = r.get(key)
    cached = cache.get(key)
    if cached:
        return json.loads(cached)
    data = fetch_fn()
    # r.setex(key, ttl, json.dumps(data))
    cache[key] = json.dumps(data)
    return data

def fetch_user():
    return {"id": 1, "name": "Alice"}

result = get_cached("user:1", fetch_user)
print("Cached:", result)

## Sessions

Store session data by session_id. Use hash (HSET) or JSON string.

In [ ]:
# r.hset("session:abc123", mapping={"user_id": "1", "role": "admin"})
# r.hgetall("session:abc123")
# r.expire("session:abc123", 3600)  # 1 hour
print("Session: hset/hgetall, expire for TTL")

## Rate Limiting

Use INCR with key per user/IP. Set expiry on first request.

In [ ]:
def rate_limit(key, limit=100, window=60):
    # count = r.incr(key)
    # if count == 1: r.expire(key, window)
    # return count <= limit
    count = cache.get(key, 0) + 1
    cache[key] = count
    return count <= limit

allowed = rate_limit("ip:192.168.1.1", limit=5)
print("Allowed:", allowed)
print("Real: INCR, EXPIRE on first request")

## Lists and Pub/Sub

Lists: LPUSH, RPOP for queues. Pub/Sub: publish/subscribe for messaging.

In [ ]:
# r.lpush("queue", "task1")
# r.rpop("queue")
# r.publish("channel", "message")
# pubsub = r.pubsub()
# pubsub.subscribe("channel")
# for msg in pubsub.listen(): ...
print("Lists: lpush/rpop. Pub/Sub: publish, subscribe")

## Connection Pooling

Use ConnectionPool for production.

In [ ]:
# pool = redis.ConnectionPool(host='localhost', port=6379, max_connections=10)
# r = redis.Redis(connection_pool=pool)
print("ConnectionPool for connection reuse")

## Common Mistakes

- **No TTL on cache**: Keys grow unbounded; use setex or expire
- **Storing large values**: Redis is in-memory; keep values small
- **Not handling connection errors**: Wrap in try/except
- **decode_responses**: Use True for string values to avoid bytes
- **Using as primary DB**: Redis can lose data; use for cache/session

## Exercises

In [ ]:
# Exercise 1: Implement cache_get(key) and cache_set(key, value, ttl=60).
# YOUR CODE HERE

In [ ]:
# Exercise 2: Write rate limiter: 10 requests per minute per user_id.
# YOUR CODE HERE

In [ ]:
# Exercise 3: Store session as JSON. session_set(sid, data), session_get(sid).
# YOUR CODE HERE

In [ ]:
# Exercise 4: Use INCR and EXPIRE. What happens if Redis restarts mid-window?
# YOUR CODE HERE

In [ ]:
# Exercise 5: Implement simple task queue with LPUSH and BRPOP (blocking pop).
# YOUR CODE HERE

In [ ]:
# Exercise 6: Cache API response. Key = f'api:{url_hash}'. Invalidate on POST.
# YOUR CODE HERE

## Key Takeaways

- Redis: set, get, setex, incr, expire
- Caching: check cache, fetch on miss, setex
- Rate limit: INCR + EXPIRE
- Sessions: hset or JSON with TTL

## 🔜 Next: Day 5 - Docker

Tomorrow: Docker basics, Dockerfile, and docker-compose for Python apps!